In [1]:
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
 
DATA_DIR = '../data/raw/'
 
# ---- Legacy file (2020-2023) ------------------------------------------------
LEGACY_FILE = '2020-2023.xlsx'
LEGACY_RAW_TABS = ['2020原始', '2021原始', '2022原始', '2023原始']
LEGACY_COA_TABS = ['COA4', 'COA5', 'COA6']
 
# ---- Modern IVL files -------------------------------------------------------
IVL_FILES = [
    '2024IVL夏季赛常规赛.xlsx',
    '2024IVL夏季赛季后赛.xlsx',
    '2024IVL秋季赛常规赛.xlsx',
    '2024IVL秋季赛季后赛.xlsx',
    '2025IVL夏季赛常规赛.xlsx',
    '2025IVL夏季赛季后赛.xlsx',
    '2025IVL秋季赛常规赛.xlsx',
    '2025IVL秋季赛季后赛.xlsx',
]
 
# ---- Modern IJL files -------------------------------------------------------
IJL_FILES = [
    '2024IJL夏季赛常规赛.xlsx',
    '2024IJL秋季赛季后赛.xlsx',
    '2025IJL夏季赛常规赛.xlsx',
    '2025IJL夏季赛季后赛.xlsx',
    '2025IJL秋季赛常规赛.xlsx',
    '2025IJL秋季赛季后赛.xlsx',
]
 
# ---- COA main event files (excluding Japan qualifiers) ----------------------
COA_FILES = [
    'COA8 全球总决赛小组赛.xlsx',
    'COA8 全球总决赛淘汰赛.xlsx',
    'COA9 全球总决赛小组赛.xlsx',
    'COA9 全球总决赛淘汰赛.xlsx',
]
 
# ---- Special / excluded -----------------------------------------------------
EXCLUDED_FILES = [
    '2025IVS.xlsx',                   # only one year, small sample
    'COA8 日本赛区预选赛.xlsx',        # regional qualifier
    'COA9 日本赛区预选赛.xlsx',        # regional qualifier
]
 
ALL_MODERN_FILES = IVL_FILES + IJL_FILES + COA_FILES
MODERN_RAW_SHEET    = '原始数据'
MODERN_PLAYER_SHEET = '赛后数据'
MODERN_GAME_SHEET   = '对局数据'
 
print("File registry loaded.")
print(f"  Legacy tabs (raw):  {LEGACY_RAW_TABS}")
print(f"  Legacy tabs (COA):  {LEGACY_COA_TABS}")
print(f"  Modern IVL files:   {len(IVL_FILES)}")
print(f"  Modern IJL files:   {len(IJL_FILES)}")
print(f"  COA main files:     {len(COA_FILES)}")
print(f"  Excluded files:     {len(EXCLUDED_FILES)}")
 
 

File registry loaded.
  Legacy tabs (raw):  ['2020原始', '2021原始', '2022原始', '2023原始']
  Legacy tabs (COA):  ['COA4', 'COA5', 'COA6']
  Modern IVL files:   8
  Modern IJL files:   6
  COA main files:     4
  Excluded files:     3


In [2]:
print("=== File existence check ===\n")
 
all_files_to_check = [LEGACY_FILE] + ALL_MODERN_FILES + EXCLUDED_FILES
 
missing = []
for f in all_files_to_check:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    status = f"OK  ({size_mb:.1f} MB)" if exists else "MISSING"
    print(f"  [{status}]  {f}")
    if not exists:
        missing.append(f)
 
if missing:
    print(f"\n⚠️  {len(missing)} file(s) missing — fix paths before continuing")
else:
    print(f"\n✅ All files found")

=== File existence check ===

  [OK  (3.1 MB)]  2020-2023.xlsx
  [OK  (1.6 MB)]  2024IVL夏季赛常规赛.xlsx
  [OK  (1.4 MB)]  2024IVL夏季赛季后赛.xlsx
  [OK  (1.8 MB)]  2024IVL秋季赛常规赛.xlsx
  [OK  (1.5 MB)]  2024IVL秋季赛季后赛.xlsx
  [OK  (2.0 MB)]  2025IVL夏季赛常规赛.xlsx
  [OK  (1.7 MB)]  2025IVL夏季赛季后赛.xlsx
  [OK  (2.0 MB)]  2025IVL秋季赛常规赛.xlsx
  [OK  (1.8 MB)]  2025IVL秋季赛季后赛.xlsx
  [OK  (1.5 MB)]  2024IJL夏季赛常规赛.xlsx
  [OK  (1.5 MB)]  2024IJL秋季赛季后赛.xlsx
  [OK  (1.9 MB)]  2025IJL夏季赛常规赛.xlsx
  [OK  (1.7 MB)]  2025IJL夏季赛季后赛.xlsx
  [OK  (1.9 MB)]  2025IJL秋季赛常规赛.xlsx
  [OK  (1.7 MB)]  2025IJL秋季赛季后赛.xlsx
  [OK  (1.8 MB)]  COA8 全球总决赛小组赛.xlsx
  [OK  (1.7 MB)]  COA8 全球总决赛淘汰赛.xlsx
  [OK  (2.2 MB)]  COA9 全球总决赛小组赛.xlsx
  [OK  (1.8 MB)]  COA9 全球总决赛淘汰赛.xlsx
  [OK  (1.7 MB)]  2025IVS.xlsx
  [OK  (1.6 MB)]  COA8 日本赛区预选赛.xlsx
  [OK  (1.8 MB)]  COA9 日本赛区预选赛.xlsx

✅ All files found


In [45]:
COLUMN_ALIASES = {
    # --- Game-level metadata ---
    '本页出错':          'page_error',
    '阶段':              'stage',
    '日期':              'date',
    '月':                'month',
    '日':                'day',
    '时间':              'time',
    '场次':              'match_num',
    '场次.1':            'match_num_2',
    '半场':              'half',
    '主场':              'home_team',
    '客场':              'away_team',
 
    # --- Hunter side ---
    '屠队主客':          'hunter_side',
    '屠队':              'hunter_team',
    '屠名':              'hunter_player',
    '屠ID':              'hunter_player',     # modern alias
    '屠选':              'hunter_character',
 
    # --- Survivor side ---
    '人队':              'survivor_team',
 
    # --- Outcome / scoring ---
    '胜利方':            'winner_side',
    '胜利':              'winner_side',
    '局分':              'half_score',
    '局总分':            'game_score',
    '场分':              'match_score',
    '主分':              'home_score',
    '客分':              'away_score',
    '剩机台数':          'gens_remaining',
    '剩机':              'gens_remaining',
 
    # --- Map / draft ---
    '地图':              'map_name',
    'BAN图':             'banned_map',
    'BAN图.1':           'banned_map_2',
    'BAN图1':            'banned_map',
    'BAN图2':            'banned_map_2',
    '地图主客':          'map_picker_side',
    '地图选择方主客':    'map_picker_side',
    '选图队伍':          'map_picker_team',
    '地图选择方队名':    'map_picker_team',
    '选边队伍':          'side_picker_team',
 
    # --- Match meta ---
    '用时 (分)':         'duration_min',
    '用时 (秒)':         'duration_sec',
    '游戏用时':          'game_duration_sec',
    'MVP':               'mvp',
    '暂停或重赛':        'paused_or_rematch',
    '备注':              'notes',
 
    # --- Hunter-side aggregate stats (modern 赛后数据) ---
    '击倒':              'hunter_knockdowns',
    '命中':              'hunter_hits',
    '震慑':              'hunter_stuns',
    '破板':              'hunter_boards_broken',
    '角色.4':            'hunter_character',     # modern 赛后: hunter character,    
    '角色.5':            'hunter_talent',
 
    # --- Match-level outcome aggregates ---
    '总逃脱':            'total_escapes',
    '逃生数':            '_qc_escape_count_check',   # underscore = ignore
    '淘汰':              'eliminations',
    '淘汰数':            'eliminations',
    '总破译进度':        'total_repair_progress',
    '总牵制时长':        'total_harassment',
 
    # --- 2020-only draft fields (pre-phased draft system) ---
    '人BAN':             'survivor_ban_2020',
    '人选':              'survivor_pick_2020',
    '屠BAN':             'hunter_ban_2020',
 
    # --- Phased draft fields (2021+) ---
    '第一阶段人BAN':     'phase1_survivor_ban',
    '第一阶段屠BAN':     'phase1_hunter_ban',
    '第一阶段人PICK':    'phase1_survivor_pick',
    '第二阶段人BAN':     'phase2_survivor_ban',
    '第二阶段人PICK':    'phase2_survivor_pick',
    '第三阶段人BAN':     'phase3_survivor_ban',
    '第三阶段人PICK':    'phase3_survivor_pick',
 
    # --- Auto-generated ban summaries (2023+) ---
    '求生者首抢禁用【自动生成】':  'survivor_first_ban_auto',
    '求生者全局禁用【自动生成】':  'survivor_global_ban_auto',
    '监管者全局禁用【自动生成】':  'hunter_global_ban_auto',
 
    # --- Talent / supplementary picks ---
    '辅助特质':          'support_talent',
    '辅助特质1':         'support_talent',
    '底牌后':            'post_trump_talent',
 
    '光明之星':          'mvp',          # 2020-only MVP-like award
 
    # --- QC / system columns (prefixed _ to flag for dropping pre-modeling) ---
    '逃脱数据出错':      '_qc_escape_error_flag',
    '逃脱数据错误':      '_qc_escape_error_flag',
}
 
# --- Survivor slot columns (1-4), built programmatically -------------------
SURVIVOR_ALIASES = {}
for i in range(1, 5):
    suffix = '' if i == 1 else f'.{i-1}'
    SURVIVOR_ALIASES[f'求生者{i}ID']        = f'survivor{i}_player'
    SURVIVOR_ALIASES[f'人ID{i}']            = f'survivor{i}_player'
    SURVIVOR_ALIASES[f'使用角色{suffix}']   = f'survivor{i}_character'
    SURVIVOR_ALIASES[f'角色{suffix}']       = f'survivor{i}_character'
    SURVIVOR_ALIASES[f'修机进度{suffix}']   = f'survivor{i}_repairs'
    SURVIVOR_ALIASES[f'修机{suffix}']       = f'survivor{i}_repairs'
    SURVIVOR_ALIASES[f'救人数{suffix}']     = f'survivor{i}_rescues'
    SURVIVOR_ALIASES[f'救人{suffix}']       = f'survivor{i}_rescues'
    SURVIVOR_ALIASES[f'治疗数{suffix}']     = f'survivor{i}_heals'
    SURVIVOR_ALIASES[f'治疗{suffix}']       = f'survivor{i}_heals'
    SURVIVOR_ALIASES[f'砸板命中{suffix}']   = f'survivor{i}_boards'
    SURVIVOR_ALIASES[f'砸板{suffix}']       = f'survivor{i}_boards'
    SURVIVOR_ALIASES[f'牵制时长{suffix}']   = f'survivor{i}_harassment'
    SURVIVOR_ALIASES[f'牵制{suffix}']       = f'survivor{i}_harassment'
    SURVIVOR_ALIASES[f'结果{suffix}']       = f'survivor{i}_result'
    SURVIVOR_ALIASES[f'淘汰{suffix}']       = f'survivor{i}_eliminated'
 
COLUMN_ALIASES.update(SURVIVOR_ALIASES)
 
print(f"Column alias dictionary built: {len(COLUMN_ALIASES)} entries")
 

Column alias dictionary built: 136 entries


In [46]:
MODERN_RAW_MARKERS = {
    '本页出错', '主场', '客场', '胜利方', '屠选', '屠名',
    '人队', '屠队', '地图', '场次'
}
MODERN_PLAYER_MARKERS = {
    '人ID1', '人ID2', '人ID3', '人ID4', '屠ID', '求生者1ID',
    '修机', '救人', '牵制', '结果', '角色',
    '主场', '客场', '场次'
}
LEGACY_MARKERS = {
    '阶段', '主场', '客场', '屠队', '地图', '胜利方', '屠选', '屠名'
}
 
def detect_header_row(path, sheet_name, markers, max_search=5):
    """Try header rows 0..max_search-1; return the row with most marker matches."""
    best_row, best_score = 0, -1
    for h in range(max_search):
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, header=h, nrows=0)
            cols = {str(c).strip() for c in df.columns}
            score = len(cols & markers)
            if score > best_score:
                best_row, best_score = h, score
        except Exception:
            continue
    return best_row, best_score
 
 
def normalize_columns(df, alias_map=COLUMN_ALIASES, verbose=False):
    """
    Rename columns using the alias map. Unknown columns left untouched.
    Returns (renamed_df, list_of_unmapped_cols).
    """
    unmapped = []
    new_names = {}
    for col in df.columns:
        col_str = str(col).strip()
        if col_str in alias_map:
            new_names[col] = alias_map[col_str]
        elif col_str.startswith('Unnamed') or col_str.startswith('_'):
            pass  # leave system columns alone
        else:
            unmapped.append(col_str)
    renamed = df.rename(columns=new_names)
    if verbose and unmapped:
        print(f"Unmapped columns: {unmapped}")
    return renamed, unmapped
 
 
def _drop_template_rows(df):
    """Drop template scaffolding rows — try multiple anchor columns."""
    # Try anchor columns in order; first one that exists is used
    for anchor in ['hunter_team', 'hunter_player', 'survivor1_player', 'home_team']:
        if anchor in df.columns:
            return df[df[anchor].notna()].copy()
    return df  # nothing matched, return as-is
 
 
def read_legacy_tab(tab_name, nrows=None, normalize=True):
    """Read a legacy 2020-2023 tab. Auto-detects header, normalizes, drops templates."""
    path = os.path.join(DATA_DIR, LEGACY_FILE)
    header_row, score = detect_header_row(path, tab_name, LEGACY_MARKERS)
    df = pd.read_excel(path, sheet_name=tab_name, header=header_row, nrows=nrows)
    
    # Legacy quirks: bare '角色' and '角色.4' both mean MVP-related fields here,
    # not survivor1_character / hunter_character like they do in modern files.
    df = df.rename(columns={
        '角色':   '_qc_mvp_character',
        '角色.4': '_qc_mvp_character_alt',
    })
    
    df['_source'] = f'legacy:{tab_name}'
    df['_header_row'] = header_row
    if normalize:
        df, _ = normalize_columns(df)
        df = _drop_template_rows(df)
    return df
 
 
def read_modern_sheet(filename, sheet_name, nrows=None, normalize=True):
    """Read a modern (2024+) sheet. Auto-detects header, normalizes, drops templates."""
    path = os.path.join(DATA_DIR, filename)
    markers = (MODERN_RAW_MARKERS if sheet_name == MODERN_RAW_SHEET
               else MODERN_PLAYER_MARKERS)
    header_row, score = detect_header_row(path, sheet_name, markers)
    if score < 2:
        print(f"⚠️  Low header detection confidence: {filename}:{sheet_name} (score={score})")
    df = pd.read_excel(path, sheet_name=sheet_name, header=header_row, nrows=nrows)
    df['_source'] = f'{filename}:{sheet_name}'
    df['_header_row'] = header_row
    if normalize:
        df, _ = normalize_columns(df)
        df = _drop_template_rows(df)
    return df
 
print("Helper functions defined.")



Helper functions defined.


In [47]:
print("=== Header row detection check ===\n")
 
print("Legacy tabs:")
for tab in LEGACY_RAW_TABS + LEGACY_COA_TABS:
    path = os.path.join(DATA_DIR, LEGACY_FILE)
    h, score = detect_header_row(path, tab, LEGACY_MARKERS)
    flag = "✅" if score >= 3 else "⚠️ "
    print(f"  {flag} {tab}: header={h} (matched {score} markers)")
 
print("\nModern files:")
for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    h_raw, s_raw = detect_header_row(path, MODERN_RAW_SHEET, MODERN_RAW_MARKERS)
    h_pl,  s_pl  = detect_header_row(path, MODERN_PLAYER_SHEET, MODERN_PLAYER_MARKERS)
    flag = "✅" if min(s_raw, s_pl) >= 3 else "⚠️ "
    print(f"  {flag} {f}")
    print(f"       原始数据 header={h_raw} ({s_raw}) · 赛后数据 header={h_pl} ({s_pl})")
 

=== Header row detection check ===

Legacy tabs:
  ✅ 2020原始: header=0 (matched 8 markers)
  ✅ 2021原始: header=0 (matched 8 markers)
  ✅ 2022原始: header=0 (matched 8 markers)
  ✅ 2023原始: header=0 (matched 8 markers)
  ✅ COA4: header=0 (matched 8 markers)
  ✅ COA5: header=0 (matched 7 markers)
  ✅ COA6: header=0 (matched 8 markers)

Modern files:
  ✅ 2024IVL夏季赛常规赛.xlsx
       原始数据 header=0 (10) · 赛后数据 header=0 (6)
  ✅ 2024IVL夏季赛季后赛.xlsx
       原始数据 header=0 (10) · 赛后数据 header=0 (6)
  ✅ 2024IVL秋季赛常规赛.xlsx
       原始数据 header=0 (9) · 赛后数据 header=1 (13)
  ✅ 2024IVL秋季赛季后赛.xlsx
       原始数据 header=0 (10) · 赛后数据 header=0 (6)
  ✅ 2025IVL夏季赛常规赛.xlsx
       原始数据 header=1 (8) · 赛后数据 header=1 (13)
  ✅ 2025IVL夏季赛季后赛.xlsx
       原始数据 header=1 (10) · 赛后数据 header=1 (13)
  ✅ 2025IVL秋季赛常规赛.xlsx
       原始数据 header=1 (8) · 赛后数据 header=1 (13)
  ✅ 2025IVL秋季赛季后赛.xlsx
       原始数据 header=1 (10) · 赛后数据 header=1 (13)
  ✅ 2024IJL夏季赛常规赛.xlsx
       原始数据 header=0 (10) · 赛后数据 header=0 (6)
  ✅ 2024IJL秋季赛季后赛.xlsx
       原始

In [48]:
print("=== Unmapped columns audit ===\n")
 
all_unmapped = set()
per_file_unmapped = {}
 
# Legacy
for tab in LEGACY_RAW_TABS + LEGACY_COA_TABS:
    df = read_legacy_tab(tab, nrows=0, normalize=False)
    _, unmapped = normalize_columns(df)
    if unmapped:
        per_file_unmapped[f'legacy:{tab}'] = unmapped
        all_unmapped.update(unmapped)
 
# Modern
for f in ALL_MODERN_FILES:
    if not os.path.exists(os.path.join(DATA_DIR, f)):
        continue
    for sheet in [MODERN_RAW_SHEET, MODERN_PLAYER_SHEET]:
        df = read_modern_sheet(f, sheet, nrows=0, normalize=False)
        _, unmapped = normalize_columns(df)
        if unmapped:
            per_file_unmapped[f'{f}:{sheet}'] = unmapped
            all_unmapped.update(unmapped)
 
print(f"=== Total unique unmapped columns: {len(all_unmapped)} ===")
for c in sorted(all_unmapped):
    print(f"  {c}")
 
if len(all_unmapped) > 15:
    print(f"\n⚠️  Many unmapped columns — extend COLUMN_ALIASES in Cell 3")
 
 

=== Unmapped columns audit ===

=== Total unique unmapped columns: 8 ===
  (底牌后)     沒带填0
  (底牌后)     没带写0
  (底牌后)   没带写0
  45
  局
  检查员用
  赛后
  赛后数据


In [50]:
print("=== Legacy raw tab schemas (canonical names) ===\n")
 
legacy_schemas = {}
for tab in LEGACY_RAW_TABS:
    df = read_legacy_tab(tab, nrows=0)
    named_cols = [c for c in df.columns
                  if not str(c).startswith('Unnamed')
                  and not str(c).startswith('_')]
    legacy_schemas[tab] = set(named_cols)
    print(f"{tab} ({len(named_cols)} mapped cols):")
    print(f"  {sorted(named_cols)}\n")
 
print("--- Columns added vs previous year ---")
tabs = LEGACY_RAW_TABS
for i in range(1, len(tabs)):
    prev, curr = tabs[i-1], tabs[i]
    added   = legacy_schemas[curr] - legacy_schemas[prev]
    removed = legacy_schemas[prev] - legacy_schemas[curr]
    print(f"\n{prev} → {curr}")
    if added:   print(f"  ADDED:   {sorted(added)}")
    if removed: print(f"  REMOVED: {sorted(removed)}")
    if not added and not removed:
        print(f"  (no changes after normalization)")
 
 

=== Legacy raw tab schemas (canonical names) ===

2020原始 (40 mapped cols):
  ['away_score', 'away_team', 'date', 'game_score', 'gens_remaining', 'half_score', 'home_score', 'home_team', 'hunter_ban_2020', 'hunter_character', 'hunter_player', 'hunter_side', 'hunter_team', 'map_name', 'map_picker_side', 'match_num', 'match_num_2', 'match_score', 'mvp', 'notes', 'stage', 'survivor1_character', 'survivor1_harassment', 'survivor1_player', 'survivor1_repairs', 'survivor2_character', 'survivor2_harassment', 'survivor2_player', 'survivor2_repairs', 'survivor3_character', 'survivor3_harassment', 'survivor3_player', 'survivor3_repairs', 'survivor4_character', 'survivor4_harassment', 'survivor4_player', 'survivor4_repairs', 'survivor_ban_2020', 'survivor_pick_2020', 'winner_side']

2021原始 (57 mapped cols):
  ['(底牌后)     沒带填0', 'away_score', 'away_team', 'banned_map', 'date', 'duration_min', 'duration_sec', 'game_score', 'gens_remaining', 'half_score', 'home_score', 'home_team', 'hunter_character'

In [51]:
print("=== Modern file schemas (canonical names) ===\n")
 
ref_file = ALL_MODERN_FILES[0]
ref_raw_df    = read_modern_sheet(ref_file, MODERN_RAW_SHEET, nrows=0)
ref_player_df = read_modern_sheet(ref_file, MODERN_PLAYER_SHEET, nrows=0)
 
def named_cols_only(df):
    return {str(c) for c in df.columns
            if not str(c).startswith('Unnamed')
            and not str(c).startswith('_')}
 
ref_raw    = named_cols_only(ref_raw_df)
ref_player = named_cols_only(ref_player_df)
 
print(f"Reference file: {ref_file}")
print(f"  原始数据 ({len(ref_raw)} cols):    {sorted(ref_raw)}")
print(f"  赛后数据 ({len(ref_player)} cols): {sorted(ref_player)}\n")
 
print("--- Schema deviations across modern files ---\n")
deviations = {}
for f in ALL_MODERN_FILES[1:]:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    raw_cols    = named_cols_only(read_modern_sheet(f, MODERN_RAW_SHEET,    nrows=0))
    player_cols = named_cols_only(read_modern_sheet(f, MODERN_PLAYER_SHEET, nrows=0))
    raw_diff    = raw_cols    ^ ref_raw
    player_diff = player_cols ^ ref_player
 
    if raw_diff or player_diff:
        print(f"⚠️  {f}")
        if raw_diff:    print(f"   原始数据 differs:  {sorted(raw_diff)}")
        if player_diff: print(f"   赛后数据 differs: {sorted(player_diff)}")
        deviations[f] = {'raw': raw_diff, 'player': player_diff}
    else:
        print(f"✅ {f}")
 
if not deviations:
    print("\n✅ All modern files have consistent schemas after normalization")
 
 

=== Modern file schemas (canonical names) ===

Reference file: 2024IVL夏季赛常规赛.xlsx
  原始数据 (32 cols):    ['away_score', 'away_team', 'banned_map', 'day', 'game_score', 'half', 'half_score', 'home_score', 'home_team', 'hunter_character', 'hunter_player', 'hunter_side', 'hunter_team', 'map_name', 'map_picker_side', 'map_picker_team', 'match_num', 'match_score', 'month', 'page_error', 'phase1_hunter_ban', 'phase1_survivor_ban', 'phase1_survivor_pick', 'phase2_survivor_ban', 'phase2_survivor_pick', 'phase3_survivor_ban', 'phase3_survivor_pick', 'side_picker_team', 'survivor_first_ban_auto', 'survivor_team', 'time', 'winner_side']
  赛后数据 (54 cols): ['away_team', 'day', 'duration_min', 'duration_sec', 'game_duration_sec', 'gens_remaining', 'half', 'home_team', 'hunter_character', 'match_num', 'match_num_2', 'month', 'mvp', 'notes', 'page_error', 'paused_or_rematch', 'survivor1_boards', 'survivor1_character', 'survivor1_harassment', 'survivor1_heals', 'survivor1_player', 'survivor1_repairs', 's

In [52]:
print("=== Column availability across eras (canonical names) ===\n")
 
cols_2020   = legacy_schemas['2020原始']
cols_2023   = legacy_schemas['2023原始']
cols_modern = ref_raw | ref_player
 
only_2023_plus = cols_2023 - cols_2020
only_modern    = cols_modern - cols_2023
 
print("Columns in 2023 but NOT in 2020 (can't use these pre-2023):")
for c in sorted(only_2023_plus):
    print(f"  {c}")
 
print(f"\nColumns new in modern format (not in 2023 raw):")
for c in sorted(only_modern):
    print(f"  {c}")

=== Column availability across eras (canonical names) ===

Columns in 2023 but NOT in 2020 (can't use these pre-2023):
  banned_map
  duration_min
  duration_sec
  half
  map_picker_team
  paused_or_rematch
  phase1_hunter_ban
  phase1_survivor_ban
  phase1_survivor_pick
  phase2_survivor_ban
  phase2_survivor_pick
  phase3_survivor_ban
  phase3_survivor_pick
  side_picker_team
  survivor1_boards
  survivor1_heals
  survivor1_rescues
  survivor1_result
  survivor2_boards
  survivor2_heals
  survivor2_rescues
  survivor2_result
  survivor3_boards
  survivor3_heals
  survivor3_rescues
  survivor3_result
  survivor4_boards
  survivor4_heals
  survivor4_rescues
  survivor4_result
  survivor_first_ban_auto
  survivor_team
  time

Columns new in modern format (not in 2023 raw):
  day
  game_duration_sec
  match_num_2
  month
  page_error
  survivor1_character
  total_escapes
  total_harassment
  total_repair_progress
  检查员用
  赛后数据


In [53]:
print("=== Row counts (after dropping template rows) ===\n")
 
total_rows = 0
 
print("Legacy tabs:")
for tab in LEGACY_RAW_TABS + LEGACY_COA_TABS:
    df = read_legacy_tab(tab)
    print(f"  {tab}: {len(df)} rows")
    total_rows += len(df)
 
print("\nModern files (原始数据):")
for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    df = read_modern_sheet(f, MODERN_RAW_SHEET)
    print(f"  {f}: {len(df)} rows")
    total_rows += len(df)
 
print(f"\nTotal rows across all included files: {total_rows}")

=== Row counts (after dropping template rows) ===

Legacy tabs:
  2020原始: 1142 rows
  2021原始: 1117 rows
  2022原始: 1117 rows
  2023原始: 1160 rows
  COA4: 90 rows
  COA5: 208 rows
  COA6: 294 rows

Modern files (原始数据):
  2024IVL夏季赛常规赛.xlsx: 496 rows
  2024IVL夏季赛季后赛.xlsx: 74 rows
  2024IVL秋季赛常规赛.xlsx: 508 rows
  2024IVL秋季赛季后赛.xlsx: 71 rows
  2025IVL夏季赛常规赛.xlsx: 502 rows
  2025IVL夏季赛季后赛.xlsx: 96 rows
  2025IVL秋季赛常规赛.xlsx: 486 rows
  2025IVL秋季赛季后赛.xlsx: 108 rows
  2024IJL夏季赛常规赛.xlsx: 230 rows
  2024IJL秋季赛季后赛.xlsx: 57 rows
  2025IJL夏季赛常规赛.xlsx: 312 rows
  2025IJL夏季赛季后赛.xlsx: 70 rows
  2025IJL秋季赛常规赛.xlsx: 304 rows
  2025IJL秋季赛季后赛.xlsx: 72 rows
  COA8 全球总决赛小组赛.xlsx: 206 rows
  COA8 全球总决赛淘汰赛.xlsx: 72 rows
  COA9 全球总决赛小组赛.xlsx: 218 rows
  COA9 全球总决赛淘汰赛.xlsx: 92 rows

Total rows across all included files: 9102


In [54]:
print("=== Missing values in key columns (modern, English names) ===\n")
 
KEY_COLS_RAW = [
    'home_team', 'away_team', 'match_num', 'half',
    'hunter_side', 'survivor_team', 'hunter_team', 'hunter_player',
    'winner_side', 'map_name', 'hunter_character', 'gens_remaining'
]
KEY_COLS_PLAYER = [
    'survivor1_player', 'survivor2_player', 'survivor3_player', 'survivor4_player',
    'survivor1_character', 'survivor1_repairs', 'survivor1_rescues',
    'survivor1_harassment', 'survivor1_result',
    'hunter_player', 'hunter_character', 'gens_remaining',
    'hunter_hits', 'hunter_knockdowns'
]
 
all_raw = [read_modern_sheet(f, MODERN_RAW_SHEET)
           for f in ALL_MODERN_FILES
           if os.path.exists(os.path.join(DATA_DIR, f))]
all_player = [read_modern_sheet(f, MODERN_PLAYER_SHEET)
              for f in ALL_MODERN_FILES
              if os.path.exists(os.path.join(DATA_DIR, f))]
 
combined_raw    = pd.concat(all_raw,    ignore_index=True)
combined_player = pd.concat(all_player, ignore_index=True)
 
print(f"Combined modern 原始数据: {len(combined_raw)} rows")
print(f"Combined modern 赛后数据: {len(combined_player)} rows\n")
 
print("Missing in 原始数据 key columns:")
for col in KEY_COLS_RAW:
    if col in combined_raw.columns:
        n = combined_raw[col].isnull().sum()
        pct = n / len(combined_raw) * 100
        flag = "⚠️ " if pct > 5 else "  "
        print(f"  {flag}{col}: {n} missing ({pct:.1f}%)")
    else:
        print(f"  ❓ {col}: NOT FOUND")
 
print("\nMissing in 赛后数据 key columns:")
for col in KEY_COLS_PLAYER:
    if col in combined_player.columns:
        n = combined_player[col].isnull().sum()
        pct = n / len(combined_player) * 100
        flag = "⚠️ " if pct > 5 else "  "
        print(f"  {flag}{col}: {n} missing ({pct:.1f}%)")
    else:
        print(f"  ❓ {col}: NOT FOUND")
 
 

=== Missing values in key columns (modern, English names) ===

Combined modern 原始数据: 3974 rows
Combined modern 赛后数据: 3960 rows

Missing in 原始数据 key columns:
    home_team: 0 missing (0.0%)
    away_team: 0 missing (0.0%)
  ⚠️ match_num: 2536 missing (63.8%)
    half: 0 missing (0.0%)
    hunter_side: 0 missing (0.0%)
    survivor_team: 0 missing (0.0%)
    hunter_team: 0 missing (0.0%)
    hunter_player: 0 missing (0.0%)
    winner_side: 0 missing (0.0%)
    map_name: 44 missing (1.1%)
    hunter_character: 16 missing (0.4%)
  ❓ gens_remaining: NOT FOUND

Missing in 赛后数据 key columns:
    survivor1_player: 4 missing (0.1%)
    survivor2_player: 16 missing (0.4%)
    survivor3_player: 194 missing (4.9%)
    survivor4_player: 22 missing (0.6%)
    survivor1_character: 64 missing (1.6%)
    survivor1_repairs: 89 missing (2.2%)
    survivor1_rescues: 81 missing (2.0%)
    survivor1_harassment: 90 missing (2.3%)
    survivor1_result: 74 missing (1.9%)
  ⚠️ hunter_player: 910 missing (23.0%)


In [55]:
print("=== Player ID audit ===\n")
 
import difflib
 
hunter_ids   = []
survivor_ids = []
 
# Pull from modern player sheets
for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    df_p = read_modern_sheet(f, MODERN_PLAYER_SHEET)
    if 'hunter_player' in df_p.columns:
        hunter_ids.extend(df_p['hunter_player'].dropna().astype(str).str.strip().tolist())
    for col in ['survivor1_player', 'survivor2_player',
                'survivor3_player', 'survivor4_player']:
        if col in df_p.columns:
            survivor_ids.extend(df_p[col].dropna().astype(str).str.strip().tolist())
 
# Pull from legacy tabs too — they have the same canonical columns now
for tab in LEGACY_RAW_TABS:
    df_l = read_legacy_tab(tab)
    if 'hunter_player' in df_l.columns:
        hunter_ids.extend(df_l['hunter_player'].dropna().astype(str).str.strip().tolist())
    for col in ['survivor1_player', 'survivor2_player',
                'survivor3_player', 'survivor4_player']:
        if col in df_l.columns:
            survivor_ids.extend(df_l[col].dropna().astype(str).str.strip().tolist())
 
unique_hunters   = sorted(set(h.lower() for h in hunter_ids   if h not in ('nan', '')))
unique_survivors = sorted(set(s.lower() for s in survivor_ids if s not in ('nan', '')))
all_players      = sorted(set(unique_hunters) | set(unique_survivors))
 
print(f"Unique hunter IDs:   {len(unique_hunters)}")
print(f"Unique survivor IDs: {len(unique_survivors)}")
print(f"Unique players total: {len(all_players)}")
 
print("\n--- Probable duplicate IDs (similarity > 0.85) ---")
duplicates_found = False
checked = set()
for name in all_players:
    if name in checked:
        continue
    candidates = [p for p in all_players if p != name and p not in checked]
    matches = difflib.get_close_matches(name, candidates, n=3, cutoff=0.85)
    if matches:
        print(f"  '{name}'  ~  {matches}")
        duplicates_found = True
    checked.add(name)
if not duplicates_found:
    print("  None found ✅")
 
# Dual-role players
dual_role = set(unique_hunters) & set(unique_survivors)
print(f"\nDual-role players (both hunter & survivor): {len(dual_role)}")
for p in sorted(dual_role):
    h_count = sum(1 for x in hunter_ids   if x.lower() == p)
    s_count = sum(1 for x in survivor_ids if x.lower() == p)
    print(f"  {p}: {h_count} hunter games, {s_count} survivor games")
 

=== Player ID audit ===

Unique hunter IDs:   100
Unique survivor IDs: 262
Unique players total: 355

--- Probable duplicate IDs (similarity > 0.85) ---
  'gua'  ~  ['guag']
  'jiu'  ~  ['jiu9']
  'lin'  ~  ['lion', 'ling']
  'lyan'  ~  ['yan']
  'mbing'  ~  ['ming']
  'mon'  ~  ['moon', 'mone']
  'nyan'  ~  ['yan']
  'pipicha'  ~  ['ppicha']
  'san'  ~  ['sans']
  'stian'  ~  ['tian']
  'xiaoci'  ~  ['xiaoi']
  'yan'  ~  ['zyan', 'yuan']
  'yuan'  ~  ['zyuan', 'yunan']
  'zyan'  ~  ['zyuan']

Dual-role players (both hunter & survivor): 7
  brontal: 55 hunter games, 4 survivor games
  chz: 88 hunter games, 7 survivor games
  musk: 9 hunter games, 11 survivor games
  ppxia: 127 hunter games, 500 survivor games
  tx: 5 hunter games, 370 survivor games
  xc: 492 hunter games, 23 survivor games
  yeyv: 53 hunter games, 74 survivor games


In [56]:
print("=== Outcome distributions ===\n")
 
if 'winner_side' in combined_raw.columns:
    print("winner_side value counts (modern):")
    print(combined_raw['winner_side'].value_counts(dropna=False))
 
    df_outcomes = combined_raw[['winner_side']].dropna()
    hunter_wins   = (df_outcomes['winner_side'] == '屠').sum()
    survivor_wins = (df_outcomes['winner_side'] == '人').sum()
    draws         = len(df_outcomes) - hunter_wins - survivor_wins
    total = len(df_outcomes)
 
    if total:
        print(f"\nHunter wins:   {hunter_wins} ({hunter_wins/total*100:.1f}%)")
        print(f"Survivor wins: {survivor_wins} ({survivor_wins/total*100:.1f}%)")
        print(f"Draws/other:   {draws} ({draws/total*100:.1f}%)")
        print(f"Total games:   {total}")
 
if 'gens_remaining' in combined_raw.columns:
    print(f"\nRemaining generators distribution:")
    print(combined_raw['gens_remaining'].value_counts().sort_index())
    impossible = combined_raw[(combined_raw['gens_remaining'] < 0) |
                               (combined_raw['gens_remaining'] > 5)]
    print(f"Rows with impossible values: {len(impossible)}")
 
 

=== Outcome distributions ===

winner_side value counts (modern):
winner_side
平     1616
屠     1550
人      807
平·       1
Name: count, dtype: int64

Hunter wins:   1550 (39.0%)
Survivor wins: 807 (20.3%)
Draws/other:   1617 (40.7%)
Total games:   3974


In [57]:
print("=== Map coverage (combined modern + legacy) ===\n")
 
# Combine all sources for coverage analysis
all_legacy = [read_legacy_tab(tab) for tab in LEGACY_RAW_TABS + LEGACY_COA_TABS]
combined_all = pd.concat(all_legacy + all_raw, ignore_index=True)
 
if 'map_name' in combined_all.columns:
    map_counts = combined_all['map_name'].value_counts()
    print(map_counts.to_string())
    low_maps = map_counts[map_counts < 20]
    if len(low_maps):
        print(f"\n⚠️  Maps with <20 appearances (unreliable fixed effects):")
        print(low_maps.to_string())
 
print("\n=== Hunter character coverage ===\n")
if 'hunter_character' in combined_all.columns:
    char_counts = combined_all['hunter_character'].value_counts()
    print(char_counts.to_string())
    low_chars = char_counts[char_counts < 20]
    if len(low_chars):
        print(f"\n⚠️  Characters with <20 appearances (unreliable baselines):")
        print(low_chars.to_string())
 

=== Map coverage (combined modern + legacy) ===

map_name
永眠镇      1782
军工厂      1378
红教堂      1251
湖景村      1130
月亮河公园    1095
唐人街      1052
里奥的回忆     692
圣心医院      515
不归林       149
-          14

⚠️  Maps with <20 appearances (unreliable fixed effects):
map_name
-    14

=== Hunter character coverage ===

hunter_character
歌剧演员     1057
梦之女巫      863
26号守卫     775
雕刻家       737
时空之影      605
红夫人       588
跛脚羊       506
破轮        457
使徒        443
喧嚣        413
守夜人       307
红蝶        304
渔女        299
蜡像师       281
蜘蛛        275
记录员       203
杂货商       166
鹿头        157
小提琴家      149
爱哭鬼       135
隐士         94
小丑         86
台球手        54
宿伞之魂       42
噩梦         24
孽蜥         24
摄影师        12
黄衣之主       10
疯眼          6
杰克          6
厂长          4
博士          3
愚人金         1

⚠️  Characters with <20 appearances (unreliable baselines):
hunter_character
摄影师     12
黄衣之主    10
疯眼       6
杰克       6
厂长       4
博士       3
愚人金      1


In [58]:
print("=== Missing values in legacy raw tabs (English names) ===\n")
 
LEGACY_KEY_COLS = {
    '2020原始': ['home_team', 'away_team', 'hunter_team', 'hunter_player',
                'winner_side', 'map_name', 'hunter_character'],
    '2021原始': ['home_team', 'away_team', 'survivor_team', 'hunter_team',
                'hunter_player', 'winner_side', 'map_name', 'hunter_character',
                'survivor1_result'],
    '2022原始': ['home_team', 'away_team', 'survivor_team', 'hunter_team',
                'hunter_player', 'winner_side', 'map_name', 'hunter_character',
                'survivor1_result'],
    '2023原始': ['home_team', 'away_team', 'survivor_team', 'hunter_team',
                'hunter_player', 'winner_side', 'map_name', 'hunter_character',
                'survivor1_result', 'survivor1_boards'],
}
 
for tab, key_cols in LEGACY_KEY_COLS.items():
    df = read_legacy_tab(tab)
    print(f"{tab} ({len(df)} rows):")
    for col in key_cols:
        if col in df.columns:
            n = df[col].isnull().sum()
            pct = n / len(df) * 100 if len(df) else 0
            flag = "⚠️ " if pct > 5 else "  "
            print(f"  {flag}{col}: {n} missing ({pct:.1f}%)")
        else:
            print(f"  ❓ {col}: NOT IN THIS TAB")
    print()
 

=== Missing values in legacy raw tabs (English names) ===

2020原始 (1142 rows):
    home_team: 0 missing (0.0%)
    away_team: 0 missing (0.0%)
    hunter_team: 0 missing (0.0%)
    hunter_player: 0 missing (0.0%)
    winner_side: 0 missing (0.0%)
    map_name: 0 missing (0.0%)
    hunter_character: 0 missing (0.0%)

2021原始 (1117 rows):
    home_team: 0 missing (0.0%)
    away_team: 0 missing (0.0%)
    survivor_team: 0 missing (0.0%)
    hunter_team: 0 missing (0.0%)
    hunter_player: 0 missing (0.0%)
    winner_side: 0 missing (0.0%)
    map_name: 0 missing (0.0%)
    hunter_character: 0 missing (0.0%)
    survivor1_result: 0 missing (0.0%)

2022原始 (1117 rows):
    home_team: 0 missing (0.0%)
    away_team: 0 missing (0.0%)
    survivor_team: 0 missing (0.0%)
    hunter_team: 0 missing (0.0%)
    hunter_player: 0 missing (0.0%)
    winner_side: 0 missing (0.0%)
    map_name: 0 missing (0.0%)
    hunter_character: 0 missing (0.0%)
    survivor1_result: 2 missing (0.2%)

2023原始 (1160 r

In [59]:
print("""
=== AUDIT SUMMARY — fill in after reviewing output above ===
 
FILE COVERAGE:
  [ ] All expected files present
  [ ] COA7 confirmed missing — noted in README
 
SCHEMA FINDINGS (after normalization):
  2020: missing half (game half), survivor_team, survivorN_boards,
        survivorN_rescues, survivorN_heals, survivorN_result
  2021+: game halves and rescues added
  2023+: board breaks and heals added
  Modern: player stats moved to separate 赛后数据 sheet
 
MODELING IMPLICATIONS:
  Bradley-Terry skill ratings:        can use ALL years
  Mixed effects variance decomp:      can use ALL years (2021+ for halves)
  Player efficiency metric:           use 2023+ only
 
PLAYER ID ISSUES FOUND:
  [ ] List confirmed duplicate IDs from Cell 12: ___
  [ ] Dual-role players found: ___
 
OUTCOME DISTRIBUTION:
  [ ] Hunter win rate: ____%
  [ ] Suspicious values: ___
 
LOW COVERAGE WARNINGS:
  [ ] Maps with <20 games: ___
  [ ] Characters with <20 games: ___
 
NEXT STEPS:
  1. Read 曾用id tab to build player alias map
  2. Apply alias normalization during ingestion
  3. Build unified panel dataset (one row per game half)
  4. Write to SQLite
""")
 


=== AUDIT SUMMARY — fill in after reviewing output above ===

FILE COVERAGE:
  [ ] All expected files present
  [ ] COA7 confirmed missing — noted in README

SCHEMA FINDINGS (after normalization):
  2020: missing half (game half), survivor_team, survivorN_boards,
        survivorN_rescues, survivorN_heals, survivorN_result
  2021+: game halves and rescues added
  2023+: board breaks and heals added
  Modern: player stats moved to separate 赛后数据 sheet

MODELING IMPLICATIONS:
  Bradley-Terry skill ratings:        can use ALL years
  Mixed effects variance decomp:      can use ALL years (2021+ for halves)
  Player efficiency metric:           use 2023+ only

PLAYER ID ISSUES FOUND:
  [ ] List confirmed duplicate IDs from Cell 12: ___
  [ ] Dual-role players found: ___

OUTCOME DISTRIBUTION:
  [ ] Hunter win rate: ____%
  [ ] Suspicious values: ___

LOW COVERAGE WARNINGS:
  [ ] Maps with <20 games: ___
  [ ] Characters with <20 games: ___

NEXT STEPS:
  1. Read 曾用id tab to build player ali

In [60]:
print("=== Per-file hunter field availability in 赛后数据 ===\n")

for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    
    # Read RAW (unnormalized) to see source column names
    df_raw = read_modern_sheet(f, MODERN_PLAYER_SHEET, normalize=False)
    
    # Check which hunter source columns exist
    hunter_id_cols = [c for c in df_raw.columns if c in ['屠ID', '屠名']]
    hunter_char_cols = [c for c in df_raw.columns if c in ['角色.4', '屠选']]
    hunter_stat_cols = [c for c in df_raw.columns if c in ['击倒', '命中']]
    
    print(f"{f}")
    print(f"  hunter ID source: {hunter_id_cols}")
    print(f"  hunter char source: {hunter_char_cols}")
    print(f"  hunter stat source: {hunter_stat_cols}")

=== Per-file hunter field availability in 赛后数据 ===

2024IVL夏季赛常规赛.xlsx
  hunter ID source: []
  hunter char source: ['角色.4']
  hunter stat source: []
2024IVL夏季赛季后赛.xlsx
  hunter ID source: []
  hunter char source: ['角色.4']
  hunter stat source: []
2024IVL秋季赛常规赛.xlsx
  hunter ID source: ['屠ID']
  hunter char source: ['角色.4']
  hunter stat source: ['命中', '击倒']
2024IVL秋季赛季后赛.xlsx
  hunter ID source: []
  hunter char source: ['角色.4']
  hunter stat source: []
2025IVL夏季赛常规赛.xlsx
  hunter ID source: ['屠ID']
  hunter char source: ['角色.4']
  hunter stat source: ['命中', '击倒']
2025IVL夏季赛季后赛.xlsx
  hunter ID source: ['屠ID']
  hunter char source: ['角色.4']
  hunter stat source: ['命中', '击倒']
2025IVL秋季赛常规赛.xlsx
  hunter ID source: ['屠ID']
  hunter char source: ['角色.4']
  hunter stat source: ['命中', '击倒']
2025IVL秋季赛季后赛.xlsx
  hunter ID source: ['屠ID']
  hunter char source: ['角色.4']
  hunter stat source: ['命中', '击倒']
2024IJL夏季赛常规赛.xlsx
  hunter ID source: []
  hunter char source: ['角色.4']
  hunter stat so